#### Import

In [1]:
import sys
from pathlib import Path
import pandas as pd
import joblib

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
)
from xgboost import XGBClassifier

sys.path.insert(0, str(Path.cwd().parent))

from src.config import GRAPH_FEATURES_CSV_PATH, SAVED_MODELS_DIR, TIME_SPLITS
from src.data.loader import load_and_prep_tabular_data

#### Load data

In [2]:
df = load_and_prep_tabular_data()

graph_df = pd.read_csv(GRAPH_FEATURES_CSV_PATH)

df = df.merge(graph_df, on="txId", how="left")
df.head()

,txId,time_step,local_0,local_1,local_2,local_3,local_4,local_5,local_6,local_7,...,agg_68,agg_69,agg_70,agg_71,class,y,in_degree,out_degree,pagerank,clustering_coefficient
0,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,-0.131155,0.677799,-0.120613,-0.119792,2,0,160,1,0.007200,0.000621
1,232029206,1,-0.005027,0.578941,-0.091383,4.380281,-0.063725,4.667146,0.851305,-0.163645,...,-0.131155,0.333211,-0.120613,-0.119792,2,0,59,1,0.001842,0.001130
2,232344069,1,-0.147852,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.137933,...,-0.131155,-0.097524,-0.120613,-0.119792,2,0,0,2,0.000041,0.000000
3,27553029,1,-0.151357,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.141519,...,-0.131155,-0.097524,-0.120613,-0.119792,2,0,1,1,0.000058,0.000000
4,3881097,1,-0.172306,-0.184668,-1.201369,0.028105,-0.043875,-0.029140,0.242712,-0.163640,...,-0.084674,-0.054450,-1.760926,-1.760984,2,0,1,1,0.000041,0.000000


#### Split data

In [3]:
train_start, train_end = TIME_SPLITS["train"]
val_start, val_end = TIME_SPLITS["val"]
test_start, test_end = TIME_SPLITS["test"]

train_df = df[(df["time_step"] >= train_start) & (df["time_step"] <= train_end)].copy()
val_df = df[(df["time_step"] >= val_start) & (df["time_step"] <= val_end)].copy()
test_df = df[(df["time_step"] >= test_start) & (df["time_step"] <= test_end)].copy()

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

Train size: 26381
Val size: 3513
Test size: 16670


#### Sanity Check

In [4]:
print("XGBoost splits:")
print(f"  Train illicit: {(train_df['y']==1).sum()}, licit: {(train_df['y']==0).sum()}")
print(f"  Val   illicit: {(val_df['y']==1).sum()}, licit: {(val_df['y']==0).sum()}")
print(f"  Test  illicit: {(test_df['y']==1).sum()}, licit: {(test_df['y']==0).sum()}")

print("\nTrain target distribution:")
print(train_df['y'].value_counts())
print(f"scale_pos_weight = {(train_df['y']==0).sum() / (train_df['y']==1).sum():.4f}")

XGBoost splits:
  Train illicit: 2871, licit: 23510
  Val   illicit: 591, licit: 2922
  Test  illicit: 1083, licit: 15587

Train target distribution:
y
0    23510
1     2871
Name: count, dtype: int64
scale_pos_weight = 8.1888


#### Define feature columns

In [5]:
exclude_cols = ["txId", "class", "y", "time_step", "in_degree", "out_degree", "pagerank", "clustering_coefficient"]
base_feature_cols = [c for c in df.columns if c not in exclude_cols]

graph_feature_cols = ["in_degree", "out_degree", "pagerank", "clustering_coefficient"]


feature_cols_A = base_feature_cols
feature_cols_B = base_feature_cols + ["time_step"]
feature_cols_C = base_feature_cols + graph_feature_cols + ["time_step"]
feature_cols_D = [c for c in df.columns if c.startswith('local_')]

print(len(feature_cols_A))
print(len(feature_cols_B))
print(len(feature_cols_C))
print(len(feature_cols_D))

165
166
170
93


#### Define one experiment

In [6]:
def run_one_experiment(train_df, val_df, test_df, feature_cols, exp_name):
    X_train, y_train = train_df[feature_cols], train_df["y"]
    X_val, y_val = val_df[feature_cols], val_df["y"]
    X_test, y_test = test_df[feature_cols], test_df["y"]

    n_pos = (y_train == 1).sum()
    n_neg = (y_train == 0).sum()
    scale_pos_weight = n_neg / n_pos

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
        early_stopping_rounds=20,
    )

    print(f"Training {exp_name}...")
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False,
    )

    y_pred = model.predict(X_test)

    recall_illicit = recall_score(y_test, y_pred, pos_label=1)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n=== {exp_name} ===")
    print(f"n_features: {len(feature_cols)}")
    print(f"Recall (illicit=1): {recall_illicit:.4f}")
    print(f"F1-macro: {f1_macro:.4f}")
    print("Confusion matrix:")
    print(cm)
    print("Classification report:")
    print(classification_report(y_test, y_pred, digits=4))

    SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    safe_name = exp_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("&", "and")
    model_path = SAVED_MODELS_DIR / f"xgboost_notebook_{safe_name}.pkl"
    joblib.dump(model, model_path)
    print(f"Model saved -> {model_path}")

    return {
        "experiment": exp_name,
        "n_features": len(feature_cols),
        "recall_illicit": recall_illicit,
        "f1_macro": f1_macro,
        "model_path": str(model_path),
    }

#### Run experiments

In [7]:
result_A = run_one_experiment(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_cols=feature_cols_A,
    exp_name="Version A (without time_step)",
)

result_B = run_one_experiment(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_cols=feature_cols_B,
    exp_name="Version B (with time_step)",
)

result_C = run_one_experiment(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_cols=feature_cols_C,
    exp_name="Version C (with Graph Features & time_step)",
)

result_D = run_one_experiment(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_cols=feature_cols_D,
    exp_name="Version D (local features only, truly graph-free)",
)

Training Version A (without time_step)...

=== Version A (without time_step) ===
n_features: 165
Recall (illicit=1): 0.7378
F1-macro: 0.8594
Confusion matrix:
[[15301   286]
 [  284   799]]
Classification report:
              precision    recall  f1-score   support

           0     0.9818    0.9817    0.9817     15587
           1     0.7364    0.7378    0.7371      1083

    accuracy                         0.9658     16670
   macro avg     0.8591    0.8597    0.8594     16670
weighted avg     0.9658    0.9658    0.9658     16670

Model saved -> D:\Document\2025.2\IT5023E_Graduation_Research_1\transaction-network-analysis\saved_models\xgboost_notebook_version_a_without_time_step.pkl
Training Version B (with time_step)...

=== Version B (with time_step) ===
n_features: 166
Recall (illicit=1): 0.7341
F1-macro: 0.8638
Confusion matrix:
[[15331   256]
 [  288   795]]
Classification report:
              precision    recall  f1-score   support

           0     0.9816    0.9836    0.9826

#### Result

In [8]:
pd.DataFrame([result_A, result_B, result_C, result_D])

,experiment,n_features,recall_illicit,f1_macro,model_path
0,Version A (without time_step),165,0.737765,0.859400,D:\Document\2025.2\IT5023E_Graduation_Research...
1,Version B (with time_step),166,0.734072,0.863824,D:\Document\2025.2\IT5023E_Graduation_Research...
2,Version C (with Graph Features & time_step),170,0.740536,0.859203,D:\Document\2025.2\IT5023E_Graduation_Research...
3,"Version D (local features only, truly graph-free)",93,0.730379,0.823407,D:\Document\2025.2\IT5023E_Graduation_Research...


#### Sanity check

In [9]:
import matplotlib.pyplot as plt

model_A = joblib.load(result_A["model_path"])

feat_imp = pd.Series(
    model_A.feature_importances_,
    index=feature_cols_A
).sort_values(ascending=False)

print("Top 20 features:")
print(feat_imp.head(20))

imp_local = feat_imp[feat_imp.index.str.startswith('local_')].sum()
imp_agg = feat_imp[feat_imp.index.str.startswith('agg_')].sum()

print(f"\nTop local vs agg breakdown:")
print(f"Local features total importance: {imp_local:.4f}")
print(f"Agg features total importance: {imp_agg:.4f}")

if imp_agg > 0.8:
    print("CẢNH BÁO: Aggregated features chiếm > 80% importance. Cần điều tra khả năng transductive leakage.")
else:
    print("Aggregated features không chiếm thế áp đảo hoàn toàn.")

Top 20 features:
local_52    0.090020
agg_9       0.075280
agg_6       0.062520
local_40    0.054902
agg_42      0.036211
local_45    0.035659
local_17    0.031546
local_89    0.031527
local_46    0.024339
agg_69      0.023720
local_4     0.022303
local_54    0.020028
agg_65      0.017343
local_58    0.017143
local_64    0.014693
local_75    0.014385
local_13    0.013669
local_79    0.012012
agg_1       0.011923
agg_67      0.011914
dtype: float32

Top local vs agg breakdown:
Local features total importance: 0.6176
Agg features total importance: 0.3824
Aggregated features không chiếm thế áp đảo hoàn toàn.
